# Train a small, fast mouse pose model (YOLO pose) on a free GPU

This notebook trains the small model that should replace the slow MARS model: **two classes (black mouse, white mouse), 7 keypoints each**.

**What it does**
1. Downloads the MARS hand-labeled top-view mouse set from Caltech Data (record 2011, **CC BY-NC 4.0, non-commercial**, 1.5 GB + 21.6 MB) and checks the checksums.
2. Turns it into a YOLO pose dataset (13,500 training pictures, 1,500 test pictures).
3. Trains `yolo26n-pose` (2.9 million parameters). About 1 to 3 hours on a free T4 GPU.
4. Measures the accuracy in pixels, with a separate look at **touching mice**.
5. Lets you download `best.pt` (about 6 MB).

**Steps:** menu **Runtime > Change runtime type > T4 GPU**, then run the cells from the top. Nothing of yours is uploaded.
A free Colab session can stop after a while. If you save to Google Drive (step 4), the results of a stopped run are not lost.

## 1. GPU and packages

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q ultralytics

## 2. Get the scripts from the project

In [ ]:
BRANCH = "ava/demo-integration"
BASE = f"https://raw.githubusercontent.com/AvaSajjadi/live-tracker/{BRANCH}/scripts"
for name in ("mars_pose_to_yolo.py", "train_yolo_pose.py", "eval_yolo_pose.py"):
    !wget -q -O {name} {BASE}/{name}
!ls -la *.py

## 3. Download and convert the data (the script checks the checksums)

In [ ]:
!mkdir -p data
!curl -L -s -o data/MARS_keypoints_top.json "https://data.caltech.edu/records/j1ww1-mdc55/files/MARS_keypoints_top.json?download=1"
!curl -L -s -o data/raw_images_top.zip "https://data.caltech.edu/records/j1ww1-mdc55/files/raw_images_top.zip?download=1"
!ls -la data
!python mars_pose_to_yolo.py --images data/raw_images_top.zip --labels data/MARS_keypoints_top.json --out /content/dataset --preview 4

## 4. (Optional) Save the results to Google Drive
If you run this, the training results are written to your Drive. Colab will ask you to allow access. Skip it if you do not want that.

In [ ]:
SAVE_TO_DRIVE = False   # change to True to save to Google Drive
PROJECT = "/content/runs"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/mouse_pose_runs"
print("results go to", PROJECT)

## 5. Train (the long step)

In [ ]:
EPOCHS = 50      # a good start; the run stops early if the test score stops improving
!python train_yolo_pose.py --data /content/dataset/dataset.yaml --epochs {EPOCHS} --imgsz 640 --batch 32 --device 0 --project {PROJECT} --name mouse_pose

## 6. How accurate is it? (pixels, in the 1024 x 570 picture)
Look at `found_share` (how often a mouse is found) and `median_error_px` (smaller is better; a mouse body is about 195 px long).
Compare **touching** with **apart**: touching mice are the hard case.

In [ ]:
!python eval_yolo_pose.py --model {PROJECT}/mouse_pose/weights/best.pt --data /content/dataset --imgsz 640 --max 600 --device 0 --out eval.json

## 7. Quick look at 6 test pictures

In [ ]:
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO
model = YOLO(f"{PROJECT}/mouse_pose/weights/best.pt")
names = [p for p in open("/content/dataset/val.txt").read().split() if p][::250][:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for ax, p in zip(axes.ravel(), names):
    r = model.predict(p, imgsz=640, conf=0.25, verbose=False)[0]
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    ax.axis("off")
plt.show()

## 8. Download the model and the accuracy numbers

In [ ]:
from google.colab import files
files.download(f"{PROJECT}/mouse_pose/weights/best.pt")
files.download("eval.json")

Next steps (on your laptop): export the model for the Intel graphics chip (`yolo export model=best.pt format=openvino imgsz=384,640`),
measure its speed and accuracy on **your own recordings** of the iPad, then plug it into the app.
The data is non-commercial: do not put the pictures or labels in the public repository.